# 00 · Data Collection (Al Jazeera Arabic)

This notebook contains the original collection and cleaning code, run in **Google Colab** in September 2026.
Outputs are cleared because they contain article text, which belongs to Al Jazeera and is not redistributed.

**Pipeline**

1. **URL + date candidates**: read the Al Jazeera sitemaps for five sections (`/sport/`, `/ebusiness/`, `/health/`, `/arts/`, `/family/`) and extract each article URL and its publication date.
2. **Feasibility of a balanced temporal split**: count articles before/after candidate cut-off dates. The *Family* section is the bottleneck (545 articles before and 363 on/after **26 Aug 2025**), so this cut-off maximizes the balanced dataset size.
3. **Candidate pool with text**: download title + body for up to 700 older and 450 newer articles per section (all available Family articles). Non-article pages (live blogs, videos, podcasts, AMP, …) and texts shorter than 200 characters are skipped. Result: **5,508** articles.
4. **Exact duplicates** (SHA-256 of normalized title + body) and **near-duplicates** (word 1–2-gram TF-IDF + cosine similarity). One exact copy and one cross-section near-duplicate pair (Health/Family, cosine 0.9245) were removed → **5,505** articles.

> To rebuild the dataset, run the cells in order in Colab. Re-scraping today may return slightly different pages if Al Jazeera has edited or removed articles.

## Step 1 · Sitemap URLs and publication dates

In [ ]:
# ============================================================
# AL JAZEERA NEWS
# STEP 1 — RECONSTRUCT URL + DATE CANDIDATES
#
# IMPORTANT:
# We freeze the dataset at 2026-09-15 so that newer articles
# added after the original collection do NOT enter the rebuild.
#
# This script DOES NOT download article content.
# It only:
#   1. Collects article URLs from Al Jazeera sitemaps
#   2. Extracts publication dates from URLs
#   3. Keeps only articles published on/before 2026-09-15
#   4. Checks date coverage for every category
#   5. Saves aljazeera_url_date_candidates.csv
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import gzip
import requests
import pandas as pd
import xml.etree.ElementTree as ET

from urllib.parse import urlparse, urljoin
from collections import OrderedDict
from datetime import date
from bisect import bisect_left


# ============================================================
# SETTINGS
# ============================================================

BASE_URL = "https://www.aljazeera.net"

# Freeze reconstruction to the same final date used originally
RECONSTRUCTION_END_DATE = date(2026, 9, 15)

# These are only used for diagnostic cutoff analysis.
# Do NOT treat the resulting cutoff as final yet.
TRAIN_TARGET = 600
TEST_TARGET = 400

OUTPUT_FILE = "aljazeera_url_date_candidates.csv"

SITEMAP_START_URLS = [
    "https://www.aljazeera.net/sitemaps/article-new.xml",
    "https://www.aljazeera.net/sitemaps/article-archive.xml",
]

SECTIONS = {
    "sport": "/sport/",
    "economy": "/ebusiness/",
    "health": "/health/",
    "arts": "/arts/",
    "family": "/family/",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ar,en-US;q=0.9,en;q=0.8",
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,*/*;q=0.8"
    ),
}


# ============================================================
# EXPECTED COUNTS FROM OUR ORIGINAL RUN
#
# These are ONLY a reconstruction check.
# If the current sitemap has changed, the values may differ.
# ============================================================

EXPECTED_COUNTS = {
    "sport": 80116,
    "economy": 78066,
    "health": 28842,
    "arts": 14010,
    "family": 908,
}


# ============================================================
# SESSION
# ============================================================

session = requests.Session()
session.headers.update(HEADERS)


# ============================================================
# NETWORK
# ============================================================

class InternetUnavailable(Exception):
    pass


def get_response(url, timeout=90, retries=4):
    """
    Request URL with retries.
    """

    last_error = None

    for attempt in range(1, retries + 1):

        try:

            response = session.get(
                url,
                timeout=timeout
            )

            if response.status_code == 200:
                return response

            if response.status_code in [
                408,
                429,
                500,
                502,
                503,
                504
            ]:

                print(
                    f"HTTP {response.status_code} "
                    f"- retry {attempt}/{retries}"
                )

                continue

            print(
                f"HTTP {response.status_code}: {url}"
            )

            return None

        except requests.RequestException as e:

            last_error = e

            print(
                f"Connection problem "
                f"- retry {attempt}/{retries}"
            )

    raise InternetUnavailable(
        f"Internet connection failed for:\n{url}\n\n"
        f"Last error: {last_error}"
    )


# ============================================================
# URL HELPERS
# ============================================================

def normalize_url(url):

    if not url:
        return None

    url = url.strip()

    if url.startswith("//"):
        url = "https:" + url

    elif url.startswith("/"):
        url = urljoin(
            BASE_URL,
            url
        )

    parsed = urlparse(url)

    clean = parsed._replace(
        query="",
        fragment=""
    ).geturl()

    if clean.endswith("/"):
        clean = clean[:-1]

    return clean


def is_valid_article_url(url, section):
    """
    Accept only standard Al Jazeera article URLs.

    Expected:
    /section/YYYY/M/D/article-slug
    """

    url = normalize_url(url)

    if not url:
        return False

    parsed = urlparse(url)

    if parsed.netloc not in {
        "www.aljazeera.net",
        "aljazeera.net",
    }:
        return False

    path = parsed.path

    prefix = SECTIONS[section]

    if not path.startswith(prefix):
        return False

    excluded = [
        "/liveblog/",
        "/video/",
        "/videos/",
        "/podcast/",
        "/programs/",
        "/infographic/",
        "/attachment/",
        "/amp/",
    ]

    lower_path = path.lower()

    if any(
        word in lower_path
        for word in excluded
    ):
        return False

    remaining = path[
        len(prefix):
    ].strip("/")

    parts = remaining.split("/")

    if len(parts) < 4:
        return False

    year = parts[0]
    month = parts[1]
    day = parts[2]

    if not (
        year.isdigit()
        and len(year) == 4
        and month.isdigit()
        and day.isdigit()
    ):
        return False

    try:

        pub_date = date(
            int(year),
            int(month),
            int(day)
        )

    except ValueError:
        return False

    # --------------------------------------------------------
    # CRITICAL RECONSTRUCTION RULE
    # Exclude anything published after 2026-09-15
    # --------------------------------------------------------

    if pub_date > RECONSTRUCTION_END_DATE:
        return False

    return True


def identify_section(url):

    for section in SECTIONS:

        if is_valid_article_url(
            url,
            section
        ):
            return section

    return None


def extract_date_from_url(url, section):
    """
    Extract publication date from URL.

    Example:
    https://www.aljazeera.net/sport/2026/8/25/xxxxx
    """

    url = normalize_url(url)

    if not url:
        return None

    parsed = urlparse(url)

    path = parsed.path

    prefix = SECTIONS[section]

    if not path.startswith(prefix):
        return None

    remaining = path[
        len(prefix):
    ].strip("/")

    parts = remaining.split("/")

    if len(parts) < 4:
        return None

    try:

        year = int(parts[0])
        month = int(parts[1])
        day = int(parts[2])

        pub_date = date(
            year,
            month,
            day
        )

    except (ValueError, TypeError):
        return None

    if pub_date > RECONSTRUCTION_END_DATE:
        return None

    return pub_date


# ============================================================
# SITEMAP PARSING
# ============================================================

def get_xml_bytes(response):

    data = response.content

    if data[:2] == b"\x1f\x8b":

        try:
            data = gzip.decompress(data)

        except Exception:
            pass

    return data


def strip_namespace(tag):

    return tag.split(
        "}",
        1
    )[-1]


def parse_sitemap(url):

    response = get_response(
        url,
        timeout=90
    )

    if response is None:
        return None, []

    try:

        root = ET.fromstring(
            get_xml_bytes(response)
        )

    except Exception as e:

        print(
            f"XML ERROR: {e}"
        )

        return None, []

    root_type = strip_namespace(
        root.tag
    )

    locations = []

    for element in root.iter():

        if (
            strip_namespace(element.tag) == "loc"
            and element.text
        ):

            locations.append(
                element.text.strip()
            )

    if root_type == "sitemapindex":
        return "index", locations

    if root_type == "urlset":
        return "urlset", locations

    return None, []


# ============================================================
# COLLECT ARTICLE URLS + DATES
# ============================================================

def collect_article_urls():

    collected = {
        section: OrderedDict()
        for section in SECTIONS
    }

    queue = list(
        SITEMAP_START_URLS
    )

    visited = set()

    print("\n" + "=" * 72)
    print("COLLECTING ARTICLE URLS + PUBLICATION DATES")
    print("=" * 72)

    print(
        f"Reconstruction frozen at: "
        f"{RECONSTRUCTION_END_DATE}"
    )

    while queue:

        sitemap_url = queue.pop(0)

        if sitemap_url in visited:
            continue

        visited.add(
            sitemap_url
        )

        print(
            f"\nSitemap {len(visited)}:"
        )

        print(
            sitemap_url
        )

        kind, locations = parse_sitemap(
            sitemap_url
        )

        if kind == "index":

            for child in locations:

                if child not in visited:
                    queue.append(
                        child
                    )

        elif kind == "urlset":

            for raw_url in locations:

                url = normalize_url(
                    raw_url
                )

                section = identify_section(
                    url
                )

                if not section:
                    continue

                publication_date = (
                    extract_date_from_url(
                        url,
                        section
                    )
                )

                if publication_date is None:
                    continue

                collected[section][
                    url
                ] = publication_date

        print(
            "Found -> "
            + " | ".join(
                f"{section}: "
                f"{len(collected[section])}"
                for section in SECTIONS
            )
        )

    result = {}

    for section in SECTIONS:

        result[section] = [
            {
                "url": url,
                "publication_date": pub_date
            }
            for url, pub_date
            in collected[section].items()
        ]

    return result


# ============================================================
# SAVE URL CANDIDATES
# ============================================================

def save_candidates(article_urls):

    rows = []

    for section in SECTIONS:

        for item in article_urls[section]:

            # Extra safety check
            if (
                item["publication_date"]
                <= RECONSTRUCTION_END_DATE
            ):

                rows.append(
                    {
                        "section": section,
                        "publication_date":
                            item["publication_date"],
                        "url": item["url"]
                    }
                )

    df = pd.DataFrame(rows)

    if not df.empty:

        df["publication_date"] = pd.to_datetime(
            df["publication_date"]
        )

        # Freeze one more time for safety
        df = df[
            df["publication_date"]
            <= pd.Timestamp("2026-09-15")
        ].copy()

        df = df.sort_values(
            [
                "section",
                "publication_date",
                "url"
            ]
        ).reset_index(
            drop=True
        )

        df.to_csv(
            OUTPUT_FILE,
            index=False,
            encoding="utf-8-sig"
        )

    return df


# ============================================================
# DATE SUMMARY
# ============================================================

def print_date_summary(article_urls):

    print("\n" + "=" * 72)
    print("DATE SUMMARY")
    print("=" * 72)

    for section in SECTIONS:

        items = article_urls[section]

        if not items:

            print(
                f"{section:8s}: "
                f"NO VALID URLS"
            )

            continue

        dates = sorted(
            item["publication_date"]
            for item in items
        )

        print(
            f"{section:8s}: "
            f"{len(dates):6d} URLs | "
            f"{dates[0]} -> {dates[-1]}"
        )


# ============================================================
# CHECK AGAINST ORIGINAL COUNTS
# ============================================================

def compare_with_original(article_urls):

    print("\n" + "=" * 72)
    print("RECONSTRUCTION CHECK")
    print("=" * 72)

    all_match = True

    for section in SECTIONS:

        current = len(
            article_urls[section]
        )

        expected = EXPECTED_COUNTS[
            section
        ]

        difference = current - expected

        match = (
            current == expected
        )

        if not match:
            all_match = False

        status = (
            "MATCH"
            if match
            else "DIFFERENT"
        )

        print(
            f"{section:8s}: "
            f"current = {current:6d} | "
            f"original = {expected:6d} | "
            f"diff = {difference:+6d} | "
            f"{status}"
        )

    print()

    if all_match:

        print(
            "All URL counts match the original reconstruction."
        )

    else:

        print(
            "WARNING: At least one URL count differs from "
            "the original run."
        )

        print(
            "DO NOT continue to article downloading yet."
        )

        print(
            "Review this output first to determine "
            "whether the sitemap changed."
        )


# ============================================================
# FIND COMMON TEMPORAL CUTOFFS
# ============================================================

def find_feasible_cutoffs(article_urls):
    """
    Find dates such that EVERY category has at least:

        TRAIN_TARGET articles BEFORE cutoff
        TEST_TARGET articles ON/AFTER cutoff
    """

    dates_by_section = {}

    all_dates = set()

    for section in SECTIONS:

        dates = sorted(
            item["publication_date"]
            for item in article_urls[section]
        )

        dates_by_section[
            section
        ] = dates

        all_dates.update(
            dates
        )

    all_dates = sorted(
        all_dates
    )

    feasible = []

    for cutoff in all_dates:

        details = {}

        valid = True

        buffers = []

        for section in SECTIONS:

            dates = dates_by_section[
                section
            ]

            index = bisect_left(
                dates,
                cutoff
            )

            older_count = index

            newer_count = (
                len(dates)
                - index
            )

            details[
                section
            ] = {
                "older": older_count,
                "newer": newer_count,
            }

            if (
                older_count < TRAIN_TARGET
                or
                newer_count < TEST_TARGET
            ):

                valid = False
                break

            buffers.append(
                min(
                    older_count
                    - TRAIN_TARGET,

                    newer_count
                    - TEST_TARGET
                )
            )

        if valid:

            feasible.append(
                {
                    "cutoff": cutoff,
                    "details": details,
                    "buffer": min(buffers)
                }
            )

    return feasible


# ============================================================
# PRINT FEASIBLE CUTOFF RESULTS
# ============================================================

def print_cutoff_results(feasible):

    print("\n" + "=" * 72)
    print("COMMON TEMPORAL CUTOFF ANALYSIS")
    print("=" * 72)

    print(
        f"\nRequired for EVERY category:"
    )

    print(
        f"Older than cutoff  : "
        f"at least {TRAIN_TARGET}"
    )

    print(
        f"On/after cutoff    : "
        f"at least {TEST_TARGET}"
    )

    if not feasible:

        print(
            "\nNO FEASIBLE COMMON CUTOFF FOUND."
        )

        return None

    print(
        f"\nNumber of feasible cutoff dates: "
        f"{len(feasible)}"
    )

    recommended = max(
        feasible,
        key=lambda x: x["buffer"]
    )

    print("\n" + "=" * 72)
    print("RECOMMENDED COMMON CUTOFF")
    print("=" * 72)

    print(
        f"\nCUTOFF DATE: "
        f"{recommended['cutoff']}"
    )

    print(
        "\nArticles BEFORE cutoff "
        "= older candidates"
    )

    print(
        "Articles ON/AFTER cutoff "
        "= newer candidates\n"
    )

    for section in SECTIONS:

        counts = (
            recommended[
                "details"
            ][section]
        )

        print(
            f"{section:8s}: "
            f"older = "
            f"{counts['older']:6d} | "
            f"newer = "
            f"{counts['newer']:6d}"
        )

    print(
        f"\nMinimum safety buffer: "
        f"{recommended['buffer']} URLs"
    )

    print("\n" + "-" * 72)

    print(
        "Earliest feasible cutoff: "
        f"{feasible[0]['cutoff']}"
    )

    print(
        "Latest feasible cutoff:   "
        f"{feasible[-1]['cutoff']}"
    )

    return recommended


# ============================================================
# MAIN
# ============================================================

try:

    article_urls = collect_article_urls()

    print_date_summary(
        article_urls
    )

    compare_with_original(
        article_urls
    )

    candidates_df = save_candidates(
        article_urls
    )

    print("\n" + "=" * 72)
    print("CANDIDATE FILE")
    print("=" * 72)

    print(
        f"\nSaved as: {OUTPUT_FILE}"
    )

    print(
        f"Rows saved: "
        f"{len(candidates_df):,}"
    )

    feasible_cutoffs = (
        find_feasible_cutoffs(
            article_urls
        )
    )

    recommended = (
        print_cutoff_results(
            feasible_cutoffs
        )
    )

    print("\n" + "=" * 72)
    print("FINISHED STEP 1")
    print("=" * 72)

    print(
        "\nIMPORTANT:"
    )

    print(
        "Do NOT start article-content downloading yet."
    )

    print(
        "Review the DATE SUMMARY, RECONSTRUCTION CHECK, "
        "and RECOMMENDED COMMON CUTOFF output first."
    )

    # --------------------------------------------------------
    # AUTOMATIC CHECKPOINT DOWNLOAD
    # --------------------------------------------------------

    try:

        from google.colab import files

        print(
            "\nDownloading checkpoint file to your computer..."
        )

        files.download(
            OUTPUT_FILE
        )

    except Exception as e:

        print(
            "\nAutomatic download was not available."
        )

        print(
            f"File is still saved as: {OUTPUT_FILE}"
        )

        print(
            f"Download message: {e}"
        )


except InternetUnavailable as e:

    print("\n" + "=" * 72)
    print("INTERNET CONNECTION LOST")
    print("=" * 72)

    print(
        str(e)
    )

## Step 2 · Largest balanced dataset for a common cut-off

In [ ]:
from bisect import bisect_left
import math

# ============================================================
# FIND THE LARGEST POSSIBLE BALANCED DATASET
# USING ONE COMMON TEMPORAL CUTOFF
# AND A 60% / 40% SPLIT
# ============================================================

TRAIN_RATIO = 0.60
TEST_RATIO = 0.40

dates_by_section = {}

all_dates = set()

for section in SECTIONS:

    dates = sorted(
        item["publication_date"]
        for item in article_urls[section]
    )

    dates_by_section[section] = dates
    all_dates.update(dates)

all_dates = sorted(all_dates)

best_result = None

for cutoff in all_dates:

    section_counts = {}
    possible_totals = []

    for section in SECTIONS:

        dates = dates_by_section[section]

        idx = bisect_left(
            dates,
            cutoff
        )

        older = idx
        newer = len(dates) - idx

        # Maximum total size this section can support
        # while preserving approximately 60/40
        max_by_train = math.floor(
            older / TRAIN_RATIO
        )

        max_by_test = math.floor(
            newer / TEST_RATIO
        )

        max_total = min(
            max_by_train,
            max_by_test
        )

        possible_totals.append(
            max_total
        )

        section_counts[section] = {
            "older_available": older,
            "newer_available": newer,
        }

    # All categories must use the SAME final size
    common_total = min(
        possible_totals
    )

    if (
        best_result is None
        or common_total > best_result["total"]
    ):

        best_result = {
            "cutoff": cutoff,
            "total": common_total,
            "counts": section_counts,
        }


# ============================================================
# CALCULATE FINAL TRAIN / TEST TARGETS
# ============================================================

TOTAL_PER_CATEGORY = best_result["total"]

TRAIN_PER_CATEGORY = round(
    TOTAL_PER_CATEGORY * TRAIN_RATIO
)

TEST_PER_CATEGORY = (
    TOTAL_PER_CATEGORY
    - TRAIN_PER_CATEGORY
)


print("=" * 72)
print("BEST COMMON TEMPORAL DATASET DESIGN")
print("=" * 72)

print(
    f"\nRecommended common cutoff: "
    f"{best_result['cutoff']}"
)

print(
    f"\nMaximum balanced size per category: "
    f"{TOTAL_PER_CATEGORY}"
)

print(
    f"Training/development per category: "
    f"{TRAIN_PER_CATEGORY}"
)

print(
    f"Temporal test per category: "
    f"{TEST_PER_CATEGORY}"
)

print(
    f"\nTotal final dataset size: "
    f"{TOTAL_PER_CATEGORY * len(SECTIONS)}"
)

print("\nAVAILABLE ARTICLES AROUND THIS CUTOFF:")
print("-" * 72)

for section in SECTIONS:

    counts = best_result[
        "counts"
    ][section]

    print(
        f"{section:8s}: "
        f"older = {counts['older_available']:6d} | "
        f"newer = {counts['newer_available']:6d}"
    )

## Step 3 · Candidate pool with article text (checkpointed to Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil

DRIVE_FOLDER = "/content/drive/MyDrive/aljazeera_thesis_data"

os.makedirs(
    DRIVE_FOLDER,
    exist_ok=True
)

source_file = "/content/aljazeera_url_date_candidates.csv"

destination_file = os.path.join(
    DRIVE_FOLDER,
    "aljazeera_url_date_candidates.csv"
)

shutil.copy2(
    source_file,
    destination_file
)

print("Saved safely to Google Drive:")
print(destination_file)

In [ ]:
# ============================================================
# AL JAZEERA CANDIDATE ARTICLE POOL
# SAFE DRIVE VERSION
#
# Reconstruct candidate pool with text.
#
# IMPORTANT:
# - Uses the same selection logic as before
# - Saves progress DIRECTLY to Google Drive
# - Can resume safely if Colab disconnects
# - Final file is also saved to Google Drive
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import time
import requests
import pandas as pd

from bs4 import BeautifulSoup


# ============================================================
# DRIVE PATHS
# ============================================================

DRIVE_FOLDER = "/content/drive/MyDrive/aljazeera_thesis_data"

os.makedirs(
    DRIVE_FOLDER,
    exist_ok=True
)

CANDIDATE_FILE = os.path.join(
    DRIVE_FOLDER,
    "aljazeera_url_date_candidates.csv"
)

PROGRESS_FILE = os.path.join(
    DRIVE_FOLDER,
    "aljazeera_pool_progress.csv"
)

FINAL_POOL_FILE = os.path.join(
    DRIVE_FOLDER,
    "aljazeera_candidate_pool_with_text.csv"
)


# ============================================================
# TEMPORARY REFERENCE DATE
# ============================================================

REFERENCE_CUTOFF = pd.Timestamp(
    "2025-08-26"
)


# ============================================================
# COLLECTION TARGETS
# ============================================================

OLDER_TARGET_OTHER = 700
NEWER_TARGET_OTHER = 450

DELAY = 1.0

# Save to Drive every 10 attempted URLs
SAVE_EVERY = 10


SECTIONS = [
    "sport",
    "economy",
    "health",
    "arts",
    "family",
]


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ar,en-US;q=0.9,en;q=0.8",
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,*/*;q=0.8"
    ),
}


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update(
    HEADERS
)


# ============================================================
# NETWORK
# ============================================================

class InternetUnavailable(Exception):
    pass


def get_response(
    url,
    timeout=45,
    retries=4
):

    last_error = None

    for attempt in range(
        1,
        retries + 1
    ):

        try:

            response = session.get(
                url,
                timeout=timeout
            )

            if response.status_code == 200:
                return response

            if response.status_code in [
                408,
                429,
                500,
                502,
                503,
                504
            ]:

                print(
                    f"   HTTP {response.status_code} "
                    f"- retry {attempt}/{retries}"
                )

                time.sleep(
                    min(
                        2 ** attempt,
                        10
                    )
                )

                continue

            print(
                f"   HTTP {response.status_code}"
            )

            return None

        except requests.RequestException as e:

            last_error = e

            print(
                f"   Connection problem "
                f"- retry {attempt}/{retries}"
            )

            time.sleep(
                min(
                    2 ** attempt,
                    10
                )
            )

    raise InternetUnavailable(
        f"Internet connection failed for:\n"
        f"{url}\n\n"
        f"Last error: {last_error}"
    )


# ============================================================
# TITLE EXTRACTION
# ============================================================

def extract_title(soup):

    if not soup:
        return ""

    h1 = soup.find("h1")

    if h1:

        title = h1.get_text(
            " ",
            strip=True
        )

        if title:
            return title

    meta = soup.find(
        "meta",
        property="og:title"
    )

    if (
        meta
        and meta.get("content")
    ):

        return meta[
            "content"
        ].strip()

    if soup.title:

        return soup.title.get_text(
            " ",
            strip=True
        )

    return ""


# ============================================================
# CONTENT EXTRACTION
# ============================================================

def extract_content(soup):

    if not soup:
        return ""

    selectors = [
        "article",
        "[class*='article-body']",
        "[class*='article-content']",
        "[class*='articleBody']",
        "[class*='articleContent']",
        "[class*='content-body']",
        "[class*='story-body']",
        "[data-testid*='article']",
    ]

    best_content = ""

    for selector in selectors:

        try:

            containers = soup.select(
                selector
            )

        except Exception:

            containers = []

        for container in containers:

            paragraphs = []

            for p in container.find_all(
                "p"
            ):

                text = p.get_text(
                    " ",
                    strip=True
                )

                if len(text) >= 20:

                    paragraphs.append(
                        text
                    )

            content = "\n".join(
                paragraphs
            )

            if (
                len(content)
                > len(best_content)
            ):

                best_content = content

    if best_content:
        return best_content

    paragraphs = []

    for p in soup.find_all("p"):

        text = p.get_text(
            " ",
            strip=True
        )

        if len(text) >= 30:

            paragraphs.append(
                text
            )

    return "\n".join(
        paragraphs
    )


# ============================================================
# ARTICLE EXTRACTION
# ============================================================

def extract_article(
    url,
    section,
    publication_date,
    temporal_side
):

    response = get_response(
        url,
        timeout=45
    )

    if response is None:
        return None

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    title = extract_title(
        soup
    )

    content = extract_content(
        soup
    )

    if not title:

        print(
            "   FAILED: no title"
        )

        return None

    if (
        not content
        or len(
            content.strip()
        ) < 200
    ):

        print(
            "   FAILED: no/short content"
        )

        return None

    return {
        "section": section,
        "title": title,
        "content": content,
        "link": url,

        "publication_date":
            pd.Timestamp(
                publication_date
            ).strftime(
                "%Y-%m-%d"
            ),

        "reference_period":
            temporal_side,
    }


# ============================================================
# LOAD URL CANDIDATES
# ============================================================

if not os.path.exists(
    CANDIDATE_FILE
):

    raise FileNotFoundError(
        "\nCould not find:\n"
        f"{CANDIDATE_FILE}\n"
    )


candidates = pd.read_csv(
    CANDIDATE_FILE
)


required_columns = {
    "section",
    "publication_date",
    "url",
}


if not required_columns.issubset(
    candidates.columns
):

    raise ValueError(
        "Candidate file must contain:\n"
        "section, publication_date, url"
    )


# ============================================================
# CLEAN CANDIDATE URL TABLE
# ============================================================

candidates[
    "publication_date"
] = pd.to_datetime(
    candidates[
        "publication_date"
    ],
    errors="coerce"
)


candidates = candidates.dropna(
    subset=[
        "section",
        "publication_date",
        "url",
    ]
)


candidates = candidates[
    candidates[
        "section"
    ].isin(
        SECTIONS
    )
]


candidates = candidates.drop_duplicates(
    subset=["url"]
)


# ============================================================
# USE FAMILY DATE RANGE AS COMMON DATE RANGE
# ============================================================

family_dates = candidates[
    candidates["section"]
    == "family"
]["publication_date"]


COMMON_START_DATE = (
    family_dates.min()
)

COMMON_END_DATE = (
    family_dates.max()
)


print("\n" + "=" * 72)
print("COMMON DATE RANGE")
print("=" * 72)

print(
    f"Start: {COMMON_START_DATE.date()}"
)

print(
    f"End:   {COMMON_END_DATE.date()}"
)

print(
    f"Temporary reference cutoff: "
    f"{REFERENCE_CUTOFF.date()}"
)


# Keep only URLs inside common date range
candidates = candidates[
    (
        candidates[
            "publication_date"
        ]
        >= COMMON_START_DATE
    )
    &
    (
        candidates[
            "publication_date"
        ]
        <= COMMON_END_DATE
    )
].copy()


# ============================================================
# ASSIGN TEMPORARY REFERENCE PERIOD
# ============================================================

candidates[
    "reference_period"
] = "older"


candidates.loc[
    candidates[
        "publication_date"
    ] >= REFERENCE_CUTOFF,
    "reference_period"
] = "newer"


# ============================================================
# SHOW AVAILABLE URL COUNTS
# ============================================================

print("\n" + "=" * 72)
print("AVAILABLE URL CANDIDATES")
print("=" * 72)

available_table = pd.crosstab(
    candidates["section"],
    candidates["reference_period"]
).reindex(
    SECTIONS,
    fill_value=0
)

print(
    available_table
)


# ============================================================
# LOAD SAVED PROGRESS FROM DRIVE
# ============================================================

if os.path.exists(
    PROGRESS_FILE
):

    progress = pd.read_csv(
        PROGRESS_FILE
    )

    print(
        f"\nLoaded previous progress from Drive: "
        f"{len(progress)} articles"
    )

else:

    progress = pd.DataFrame(
        columns=[
            "section",
            "title",
            "content",
            "link",
            "publication_date",
            "reference_period",
        ]
    )

    print(
        "\nNo previous progress found. "
        "Starting clean reconstruction."
    )


progress = progress.drop_duplicates(
    subset=["link"]
)


records = progress.to_dict(
    "records"
)


existing_links = set(
    progress["link"].astype(str)
)


# ============================================================
# SAVE PROGRESS DIRECTLY TO DRIVE
# ============================================================

def save_progress(records):

    df = pd.DataFrame(
        records
    )

    if not df.empty:

        df = df.drop_duplicates(
            subset=["link"]
        )

        df = df.sort_values(
            [
                "section",
                "publication_date",
            ]
        ).reset_index(
            drop=True
        )

    df.to_csv(
        PROGRESS_FILE,
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# CURRENT COUNT
# ============================================================

def current_count(
    records,
    section,
    temporal_side
):

    return sum(
        1
        for record in records
        if (
            record["section"]
            == section
            and
            record[
                "reference_period"
            ]
            == temporal_side
        )
    )


# ============================================================
# COLLECT ONE GROUP
# ============================================================

def collect_group(
    section,
    temporal_side,
    target=None,
    collect_all=False
):

    global records
    global existing_links

    subset = candidates[
        (
            candidates["section"]
            == section
        )
        &
        (
            candidates[
                "reference_period"
            ]
            == temporal_side
        )
    ].copy()


    # SAME deterministic seeds as original run
    section_seed = {
        "sport": 101,
        "economy": 202,
        "health": 303,
        "arts": 404,
        "family": 505,
    }[section]


    if temporal_side == "newer":
        section_seed += 1000


    subset = subset.sample(
        frac=1,
        random_state=section_seed
    ).reset_index(
        drop=True
    )


    current = current_count(
        records,
        section,
        temporal_side
    )


    if collect_all:

        target_display = len(
            subset
        )

    else:

        target_display = target


    print("\n" + "=" * 72)

    print(
        f"{section.upper()} | "
        f"{temporal_side.upper()}"
    )

    print("=" * 72)

    print(
        f"Already collected: "
        f"{current}"
    )


    if collect_all:

        print(
            f"Available URLs to attempt: "
            f"{len(subset)}"
        )

    else:

        print(
            f"Successful article target: "
            f"{target}"
        )


    processed = 0


    for _, row in subset.iterrows():

        if (
            not collect_all
            and current >= target
        ):
            break


        url = str(
            row["url"]
        )


        if url in existing_links:
            continue


        processed += 1


        print(
            f"\n{section} | "
            f"{temporal_side} | "
            f"collected={current}"
        )

        print(
            row[
                "publication_date"
            ].date(),
            url
        )


        article = extract_article(
            url=url,
            section=section,
            publication_date=
                row[
                    "publication_date"
                ],
            temporal_side=
                temporal_side,
        )


        if article:

            records.append(
                article
            )

            existing_links.add(
                url
            )

            current += 1

            print(
                f"   OK -> "
                f"{current}"
            )


        if (
            processed
            % SAVE_EVERY
            == 0
        ):

            save_progress(
                records
            )

            print(
                f"   [Checkpoint saved to Drive | "
                f"{len(records)} total articles]"
            )


        time.sleep(
            DELAY
        )


    # Save when each group finishes
    save_progress(
        records
    )


    print(
        f"\nFinished {section} "
        f"{temporal_side}: "
        f"{current} successful articles"
    )


# ============================================================
# MAIN COLLECTION
# ============================================================

try:

    for section in SECTIONS:

        # FAMILY:
        # attempt every available URL
        if section == "family":

            collect_group(
                section="family",
                temporal_side="older",
                collect_all=True,
            )

            collect_group(
                section="family",
                temporal_side="newer",
                collect_all=True,
            )

        # OTHER CATEGORIES
        else:

            collect_group(
                section=section,
                temporal_side="older",
                target=OLDER_TARGET_OTHER,
                collect_all=False,
            )

            collect_group(
                section=section,
                temporal_side="newer",
                target=NEWER_TARGET_OTHER,
                collect_all=False,
            )


except InternetUnavailable as e:

    save_progress(
        records
    )

    print("\n" + "=" * 72)
    print("INTERNET CONNECTION LOST")
    print("=" * 72)

    print(
        str(e)
    )

    print(
        "\nProgress was saved safely to Google Drive."
    )

    print(
        "Reconnect and run THIS SAME CELL again."
    )


# ============================================================
# FINALIZE CANDIDATE POOL
# ============================================================

pool_df = pd.DataFrame(
    records
)


if not pool_df.empty:

    pool_df = pool_df.drop_duplicates(
        subset=["link"]
    )


    pool_df[
        "publication_date"
    ] = pd.to_datetime(
        pool_df[
            "publication_date"
        ]
    )


    pool_df = pool_df.sort_values(
        [
            "section",
            "publication_date",
        ]
    ).reset_index(
        drop=True
    )


    pool_df.to_csv(
        FINAL_POOL_FILE,
        index=False,
        encoding="utf-8-sig"
    )


    print("\n" + "=" * 72)
    print("CANDIDATE ARTICLE POOL CREATED")
    print("=" * 72)


    print(
        f"\nFile saved safely on Drive:\n"
        f"{FINAL_POOL_FILE}"
    )


    print(
        f"\nTotal successfully downloaded: "
        f"{len(pool_df)}"
    )


    print(
        "\nCOUNTS BY CATEGORY AND REFERENCE PERIOD"
    )

    print("-" * 72)


    summary = pd.crosstab(
        pool_df[
            "section"
        ],
        pool_df[
            "reference_period"
        ]
    ).reindex(
        SECTIONS,
        fill_value=0
    )


    print(
        summary
    )


    print(
        "\nDATE RANGES"
    )

    print("-" * 72)


    date_summary = (
        pool_df
        .groupby(
            [
                "section",
                "reference_period",
            ]
        )[
            "publication_date"
        ]
        .agg(
            [
                "min",
                "max",
                "count",
            ]
        )
    )


    print(
        date_summary
    )


    print("\nIMPORTANT:")
    print(
        "reference_period is NOT the final train/test split."
    )

    print(
        "Next step: verify reconstruction against the "
        "250 audited articles BEFORE duplicate cleaning."
    )


    # ========================================================
    # DOWNLOAD FINAL FILE TO COMPUTER AS EXTRA BACKUP
    # ========================================================

    try:

        from google.colab import files

        print(
            "\nDownloading an extra copy to your computer..."
        )

        files.download(
            FINAL_POOL_FILE
        )

    except Exception as e:

        print(
            "\nAutomatic computer download did not run."
        )

        print(
            "The complete file is still safely saved "
            "on Google Drive."
        )

        print(
            e
        )


else:

    print(
        "\nNo articles were downloaded."
    )

## Step 4a · Exact duplicates + near-duplicate audit

Reads `aljazeera_candidate_pool_with_text.csv` from the working directory (copy it from the Drive folder if the runtime was restarted).

In [ ]:
# ============================================================
# STEP 2A — EXACT DUPLICATES + COSINE NEAR-DUPLICATE AUDIT
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata
import hashlib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# ------------------------------------------------------------
# FILES
# ------------------------------------------------------------
INPUT_FILE = "aljazeera_candidate_pool_with_text.csv"

EXACT_CLEAN_FILE = "aljazeera_after_exact_dedup.csv"
EXACT_DUP_FILE = "aljazeera_exact_duplicates_removed.csv"
CROSS_LABEL_EXACT_FILE = "aljazeera_exact_cross_label_review.csv"
NEAR_DUP_FILE = "aljazeera_near_duplicate_candidates.csv"

# We deliberately use a broad threshold for AUDITING first.
# Final clustering threshold will be chosen after inspecting results.
AUDIT_THRESHOLD = 0.90

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
df = pd.read_csv(INPUT_FILE)

print("=" * 72)
print("INPUT DATA")
print("=" * 72)
print(f"Rows loaded: {len(df):,}")
print()

required_cols = ["section", "title", "content", "link", "publication_date"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["title"] = df["title"].fillna("").astype(str)
df["content"] = df["content"].fillna("").astype(str)
df["section"] = df["section"].astype(str)
df["publication_date"] = pd.to_datetime(
    df["publication_date"], errors="coerce"
)

# ------------------------------------------------------------
# CONSERVATIVE NORMALIZATION FOR DUPLICATE DETECTION
# ------------------------------------------------------------
arabic_diacritics = re.compile(
    r"""
    ّ    | # Shadda
    َ    | # Fatha
    ً    | # Tanwin Fath
    ُ    | # Damma
    ٌ    | # Tanwin Damm
    ِ    | # Kasra
    ٍ    | # Tanwin Kasr
    ْ    | # Sukun
    ـ      # Tatweel
    """,
    re.VERBOSE
)

def normalize_for_dedup(text):
    text = "" if pd.isna(text) else str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Remove zero-width / invisible characters
    text = re.sub(r"[\u200b-\u200f\u202a-\u202e\u2060\ufeff]", " ", text)

    # Remove Arabic diacritics and tatweel
    text = re.sub(arabic_diacritics, "", text)

    # Lowercase Latin text if present
    text = text.lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Use TITLE + CONTENT as requested in the review
df["dedup_text"] = (
    df["title"].map(normalize_for_dedup)
    + " "
    + df["content"].map(normalize_for_dedup)
).str.strip()

# ------------------------------------------------------------
# EXACT HASH
# ------------------------------------------------------------
df["exact_hash"] = df["dedup_text"].apply(
    lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()
)

hash_counts = df["exact_hash"].value_counts()
duplicate_hashes = hash_counts[hash_counts > 1].index

exact_dup_rows = df[df["exact_hash"].isin(duplicate_hashes)].copy()

# Detect exact duplicate groups that appear under different labels
cross_label_hashes = []

for h, group in exact_dup_rows.groupby("exact_hash"):
    if group["section"].nunique() > 1:
        cross_label_hashes.append(h)

cross_label_exact = df[df["exact_hash"].isin(cross_label_hashes)].copy()

# Sort so the earliest article is retained as representative
df = df.sort_values(
    ["publication_date", "section", "link"],
    na_position="last"
).reset_index(drop=True)

# Keep one representative for each exact normalized document
exact_clean = df.drop_duplicates(
    subset=["exact_hash"],
    keep="first"
).copy()

# Rows removed by exact deduplication
kept_indices = set(exact_clean.index)

# Easier: compare links retained vs original
kept_links = set(exact_clean["link"])
removed_exact = df[~df["link"].isin(kept_links)].copy()

# ------------------------------------------------------------
# SAVE EXACT-DUPLICATE RESULTS
# ------------------------------------------------------------
exact_clean.to_csv(EXACT_CLEAN_FILE, index=False)
removed_exact.to_csv(EXACT_DUP_FILE, index=False)
cross_label_exact.to_csv(CROSS_LABEL_EXACT_FILE, index=False)

print("=" * 72)
print("EXACT DUPLICATE RESULTS")
print("=" * 72)
print(f"Original rows:                 {len(df):,}")
print(f"Unique normalized documents:  {len(exact_clean):,}")
print(f"Exact duplicate rows removed: {len(df) - len(exact_clean):,}")
print(f"Exact duplicate groups:       {len(duplicate_hashes):,}")
print(f"Cross-label exact groups:     {len(cross_label_hashes):,}")
print()

print("COUNTS AFTER EXACT DEDUP")
print("-" * 72)
print(exact_clean["section"].value_counts().sort_index())
print()

# ------------------------------------------------------------
# TF-IDF FOR NEAR-DUPLICATE AUDIT
# ------------------------------------------------------------
print("=" * 72)
print("BUILDING TF-IDF FOR NEAR-DUPLICATE DETECTION")
print("=" * 72)

vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    max_features=120000,
    sublinear_tf=True,
    norm="l2"
)

X = vectorizer.fit_transform(exact_clean["dedup_text"])

print(f"Documents: {X.shape[0]:,}")
print(f"TF-IDF features: {X.shape[1]:,}")
print()

# ------------------------------------------------------------
# COSINE NEIGHBOR SEARCH
# distance = 1 - cosine similarity
# ------------------------------------------------------------
radius = 1.0 - AUDIT_THRESHOLD

nn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

nn.fit(X)

distances, indices = nn.radius_neighbors(
    X,
    radius=radius,
    return_distance=True,
    sort_results=True
)

pairs = []

for i, (dists, inds) in enumerate(zip(distances, indices)):
    for dist, j in zip(dists, inds):

        # Avoid self-pair and duplicate pair direction
        if j <= i:
            continue

        similarity = 1.0 - float(dist)

        row_i = exact_clean.iloc[i]
        row_j = exact_clean.iloc[j]

        pairs.append({
            "similarity": similarity,

            "section_1": row_i["section"],
            "section_2": row_j["section"],

            "date_1": row_i["publication_date"],
            "date_2": row_j["publication_date"],

            "title_1": row_i["title"],
            "title_2": row_j["title"],

            "link_1": row_i["link"],
            "link_2": row_j["link"],

            "same_section": row_i["section"] == row_j["section"]
        })

near_df = pd.DataFrame(pairs)

if len(near_df) > 0:
    near_df = near_df.sort_values(
        "similarity",
        ascending=False
    ).reset_index(drop=True)

near_df.to_csv(NEAR_DUP_FILE, index=False)

# ------------------------------------------------------------
# SUMMARY AT SEVERAL THRESHOLDS
# ------------------------------------------------------------
print("=" * 72)
print("COSINE NEAR-DUPLICATE AUDIT")
print("=" * 72)

if len(near_df) == 0:
    print(f"No pairs found with cosine similarity >= {AUDIT_THRESHOLD:.2f}")

else:
    for threshold in [0.90, 0.92, 0.95, 0.97]:
        count = (near_df["similarity"] >= threshold).sum()

        cross_label_count = (
            (near_df["similarity"] >= threshold)
            & (~near_df["same_section"])
        ).sum()

        print(
            f">= {threshold:.2f}: "
            f"{count:,} pairs "
            f"| cross-label: {cross_label_count:,}"
        )

    print()
    print("TOP 20 MOST SIMILAR PAIRS")
    print("-" * 72)

    display_cols = [
        "similarity",
        "section_1",
        "section_2",
        "date_1",
        "date_2",
        "title_1",
        "title_2"
    ]

    print(
        near_df[display_cols]
        .head(20)
        .to_string(index=False)
    )

print()
print("=" * 72)
print("FILES CREATED")
print("=" * 72)
print(EXACT_CLEAN_FILE)
print(EXACT_DUP_FILE)
print(CROSS_LABEL_EXACT_FILE)
print(NEAR_DUP_FILE)

print()
print("IMPORTANT:")
print("No final temporal split has been created yet.")
print("Near-duplicate candidates have NOT been deleted yet.")
print("Next step: inspect similarity counts, choose the final")
print("near-duplicate threshold, create clusters, then recalculate")
print("the common temporal cutoff.")

## Step 4b · Remove the confirmed cross-section near-duplicate pair

In [ ]:
import pandas as pd

# ------------------------------------------------------------
# FILES
# ------------------------------------------------------------
INPUT_FILE = "aljazeera_after_exact_dedup.csv"
NEAR_DUP_FILE = "aljazeera_near_duplicate_candidates.csv"

OUTPUT_FILE = "aljazeera_after_duplicate_cleaning.csv"
REMOVED_FILE = "aljazeera_cross_label_near_duplicates_removed.csv"

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
df = pd.read_csv(INPUT_FILE)
near_df = pd.read_csv(NEAR_DUP_FILE)

# Select cross-label near-duplicate pairs with similarity >= 0.90
pairs_to_remove = near_df[
    (near_df["similarity"] >= 0.90) &
    (near_df["section_1"] != near_df["section_2"])
].copy()

# Collect both URLs from each selected pair
links_to_remove = set(pairs_to_remove["link_1"]).union(
    set(pairs_to_remove["link_2"])
)

# Save removed articles for documentation
removed = df[df["link"].isin(links_to_remove)].copy()

# Final cleaned candidate pool
cleaned = df[~df["link"].isin(links_to_remove)].copy()

# Save
removed.to_csv(REMOVED_FILE, index=False)
cleaned.to_csv(OUTPUT_FILE, index=False)

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------
print("=" * 72)
print("NEAR-DUPLICATE REMOVAL COMPLETED")
print("=" * 72)

print(f"Rows before removal: {len(df):,}")
print(f"Rows removed:        {len(removed):,}")
print(f"Rows remaining:      {len(cleaned):,}")

print()
print("REMOVED ARTICLES")
print("-" * 72)

print(
    removed[
        ["section", "publication_date", "title", "link"]
    ].to_string(index=False)
)

print()
print("COUNTS AFTER DUPLICATE CLEANING")
print("-" * 72)

print(
    cleaned["section"]
    .value_counts()
    .sort_index()
)

print()
print("Saved cleaned dataset:")
print(OUTPUT_FILE)

print()
print("Saved removed articles for documentation:")
print(REMOVED_FILE)